# Script 2.1 — Análise Exploratória de Dados Pós-Processamento
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

Roda após o Script 2 e antes do Script 3. Gera um relatório visual completo
do dataset final para documentação no TCC.

| Bloco | Conteúdo |
|-------|----------|
| 1 | Visão geral: shape, empresas, cobertura temporal |
| 2 | Distribuição dos targets (log-transform justificado) |
| 3 | Evolução temporal dos KPIs por setor |
| 4 | Heatmap de correlação das features com os targets |
| 5 | Matriz de missingness das features |
| 6 | Cobertura das variáveis macro por empresa e ano |
| 7 | Análise de outliers por KPI e setor |
| 8 | Distribuição setorial das observações |
| 9 | Estatísticas descritivas consolidadas para o TCC |
| 10 | Painel longitudinal por empresa — KPIs + Receita/Lucro/EBITDA (2015–2025) |

## Etapa 0 — Dependências e configuração

In [ ]:
import logging, warnings, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

PASTA_SAIDA = Path('outputs')
PASTA_EDA   = PASTA_SAIDA / 'eda'
PASTA_EDA.mkdir(exist_ok=True)

# Paleta consistente com o TCC
CORES_SETOR = {
    'Petróleo':    '#1f4e79',
    'Energia':     '#2e75b6',
    'Varejo':      '#ed7d31',
    'Commodities': '#70ad47',
    'Tecnologia':  '#ffc000',
}
CORES_ORIGEM = {'DFP': '#1f4e79', 'ITR': '#ed7d31'}

logger = logging.getLogger('eda')
logger.setLevel(logging.INFO)
logger.handlers.clear()
_sh = logging.StreamHandler()
_sh.setFormatter(logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                                    datefmt='%H:%M:%S'))
logger.addHandler(_sh)

logger.info("Script 2.1 — EDA iniciado")
print("✅ Dependências carregadas")


13:36:38 | INFO     | Script 2.1 — EDA iniciado


✅ Dependências carregadas


## Etapa 1 — Carregamento

In [ ]:
import pickle

dataset = pd.read_parquet(PASTA_SAIDA / 'dataset_preparado.parquet')
ds_raw  = pd.read_parquet(PASTA_SAIDA / 'dataset_cvm_consolidado.parquet')

with open(PASTA_SAIDA / 'features.pkl',  'rb') as f: FEATURES = pickle.load(f)
with open(PASTA_SAIDA / 'targets.pkl',   'rb') as f: TARGETS  = pickle.load(f)
with open(PASTA_SAIDA / 'kpis.pkl',      'rb') as f: KPIS     = pickle.load(f)

MACRO_COLS = [c for c in dataset.columns
              if c.startswith('macro_') or c in ['retorno_12m','volatilidade_60d']]
SETOR_COLS = [c for c in dataset.columns if c.startswith('setor_')]
YOY_COLS   = [c for c in dataset.columns if c.endswith('_yoy')]

# Reconstruir SETOR a partir do one-hot
setor_map = {c: c.replace('setor_','') for c in SETOR_COLS}
dataset['SETOR'] = (dataset[SETOR_COLS]
                    .idxmax(axis=1)
                    .map(setor_map))

logger.info("Dataset: %d × %d | Empresas: %d | Features: %d | Targets: %s",
            *dataset.shape, dataset['NOME_CIA'].nunique(), len(FEATURES), TARGETS)
print(f"Dataset: {dataset.shape} | Empresas: {dataset['NOME_CIA'].nunique()} | "
      f"Features: {len(FEATURES)} | Targets: {TARGETS}")


13:36:43 | INFO     | Dataset: 1031 × 992 | Empresas: 25 | Features: 415 | Targets: ['TARGET_DRE_3.01_ITR_T1', 'TARGET_DRE_3.01_ITR_T2', 'TARGET_DRE_3.01_ITR_T3', 'TARGET_DRE_3.01_DFP', 'TARGET_DRE_3.11_ITR_T1', 'TARGET_DRE_3.11_ITR_T2', 'TARGET_DRE_3.11_ITR_T3', 'TARGET_DRE_3.11_DFP', 'TARGET_EBITDA_ITR_T1', 'TARGET_EBITDA_ITR_T2', 'TARGET_EBITDA_ITR_T3', 'TARGET_EBITDA_DFP', 'TARGET_BPA_1_ITR_T1', 'TARGET_BPA_1_ITR_T2', 'TARGET_BPA_1_ITR_T3', 'TARGET_BPA_1_DFP', 'TARGET_BPA_1.01_ITR_T1', 'TARGET_BPA_1.01_ITR_T2', 'TARGET_BPA_1.01_ITR_T3', 'TARGET_BPA_1.01_DFP', 'TARGET_BPP_2.01_ITR_T1', 'TARGET_BPP_2.01_ITR_T2', 'TARGET_BPP_2.01_ITR_T3', 'TARGET_BPP_2.01_DFP', 'TARGET_BPP_2.03_ITR_T1', 'TARGET_BPP_2.03_ITR_T2', 'TARGET_BPP_2.03_ITR_T3', 'TARGET_BPP_2.03_DFP', 'TARGET_BPP_2_ITR_T1', 'TARGET_BPP_2_ITR_T2', 'TARGET_BPP_2_ITR_T3', 'TARGET_BPP_2_DFP', 'TARGET_DFC_MI_6.01_ITR_T1', 'TARGET_DFC_MI_6.01_ITR_T2', 'TARGET_DFC_MI_6.01_ITR_T3', 'TARGET_DFC_MI_6.01_DFP']


Dataset: (1031, 992) | Empresas: 25 | Features: 415 | Targets: ['TARGET_DRE_3.01_ITR_T1', 'TARGET_DRE_3.01_ITR_T2', 'TARGET_DRE_3.01_ITR_T3', 'TARGET_DRE_3.01_DFP', 'TARGET_DRE_3.11_ITR_T1', 'TARGET_DRE_3.11_ITR_T2', 'TARGET_DRE_3.11_ITR_T3', 'TARGET_DRE_3.11_DFP', 'TARGET_EBITDA_ITR_T1', 'TARGET_EBITDA_ITR_T2', 'TARGET_EBITDA_ITR_T3', 'TARGET_EBITDA_DFP', 'TARGET_BPA_1_ITR_T1', 'TARGET_BPA_1_ITR_T2', 'TARGET_BPA_1_ITR_T3', 'TARGET_BPA_1_DFP', 'TARGET_BPA_1.01_ITR_T1', 'TARGET_BPA_1.01_ITR_T2', 'TARGET_BPA_1.01_ITR_T3', 'TARGET_BPA_1.01_DFP', 'TARGET_BPP_2.01_ITR_T1', 'TARGET_BPP_2.01_ITR_T2', 'TARGET_BPP_2.01_ITR_T3', 'TARGET_BPP_2.01_DFP', 'TARGET_BPP_2.03_ITR_T1', 'TARGET_BPP_2.03_ITR_T2', 'TARGET_BPP_2.03_ITR_T3', 'TARGET_BPP_2.03_DFP', 'TARGET_BPP_2_ITR_T1', 'TARGET_BPP_2_ITR_T2', 'TARGET_BPP_2_ITR_T3', 'TARGET_BPP_2_DFP', 'TARGET_DFC_MI_6.01_ITR_T1', 'TARGET_DFC_MI_6.01_ITR_T2', 'TARGET_DFC_MI_6.01_ITR_T3', 'TARGET_DFC_MI_6.01_DFP']


## Bloco 1 — Visão Geral

In [ ]:
print("="*70)
print("  BLOCO 1 — Visão Geral do Dataset Final")
print("="*70)

origens = dataset['ORIGEM'].value_counts()
anos    = sorted(dataset['ANO'].dropna().astype(int).unique())

print(f"  Observações : {len(dataset):,} total  "
      f"(DFP: {origens.get('DFP',0)} | ITR: {origens.get('ITR',0)})")
print(f"  Empresas    : {dataset['NOME_CIA'].nunique()} / 25")
print(f"  Setores     : {dataset['SETOR'].nunique()}")
print(f"  Período     : {anos[0]} – {anos[-1]}  ({len(anos)} anos)")
print(f"  Features ML : {len(FEATURES)}")
print(f"  Targets     : {TARGETS}")
print(f"  Macro vars  : {MACRO_COLS}")

# ── Figura 1: obs por ano e origem ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras empilhadas por ano
pivot = (dataset.groupby(['ANO','ORIGEM'])
         .size().unstack(fill_value=0).reset_index())
pivot['ANO'] = pivot['ANO'].astype(int)
x = np.arange(len(pivot))
w = 0.6
axes[0].bar(x, pivot.get('DFP', 0), w, label='DFP', color=CORES_ORIGEM['DFP'])
axes[0].bar(x, pivot.get('ITR', 0), w,
            bottom=pivot.get('DFP', 0), label='ITR', color=CORES_ORIGEM['ITR'])
axes[0].set_xticks(x)
axes[0].set_xticklabels(pivot['ANO'].astype(str), rotation=45)
axes[0].set_title('Observações por Ano e Origem', fontsize=12, fontweight='bold')
axes[0].set_ylabel('N° de observações')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Pizza por setor
setor_cnt = dataset['SETOR'].value_counts()
cores_lista = [CORES_SETOR.get(s, '#999') for s in setor_cnt.index]
axes[1].pie(setor_cnt.values, labels=setor_cnt.index,
            colors=cores_lista, autopct='%1.0f%%',
            startangle=90, pctdistance=0.8)
axes[1].set_title('Distribuição por Setor', fontsize=12, fontweight='bold')

plt.suptitle('Composição do Dataset Final', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PASTA_EDA / 'b1_composicao_dataset.png', dpi=150, bbox_inches='tight')
plt.close()
logger.info("Bloco 1 salvo")
print("  ✅ Figura salva: b1_composicao_dataset.png")


  BLOCO 1 — Visão Geral do Dataset Final
  Observações : 1,031 total  (DFP: 254 | ITR: 777)
  Empresas    : 25 / 25
  Setores     : 5
  Período     : 2015 – 2026  (12 anos)
  Features ML : 415
  Targets     : ['TARGET_DRE_3.01_ITR_T1', 'TARGET_DRE_3.01_ITR_T2', 'TARGET_DRE_3.01_ITR_T3', 'TARGET_DRE_3.01_DFP', 'TARGET_DRE_3.11_ITR_T1', 'TARGET_DRE_3.11_ITR_T2', 'TARGET_DRE_3.11_ITR_T3', 'TARGET_DRE_3.11_DFP', 'TARGET_EBITDA_ITR_T1', 'TARGET_EBITDA_ITR_T2', 'TARGET_EBITDA_ITR_T3', 'TARGET_EBITDA_DFP', 'TARGET_BPA_1_ITR_T1', 'TARGET_BPA_1_ITR_T2', 'TARGET_BPA_1_ITR_T3', 'TARGET_BPA_1_DFP', 'TARGET_BPA_1.01_ITR_T1', 'TARGET_BPA_1.01_ITR_T2', 'TARGET_BPA_1.01_ITR_T3', 'TARGET_BPA_1.01_DFP', 'TARGET_BPP_2.01_ITR_T1', 'TARGET_BPP_2.01_ITR_T2', 'TARGET_BPP_2.01_ITR_T3', 'TARGET_BPP_2.01_DFP', 'TARGET_BPP_2.03_ITR_T1', 'TARGET_BPP_2.03_ITR_T2', 'TARGET_BPP_2.03_ITR_T3', 'TARGET_BPP_2.03_DFP', 'TARGET_BPP_2_ITR_T1', 'TARGET_BPP_2_ITR_T2', 'TARGET_BPP_2_ITR_T3', 'TARGET_BPP_2_DFP', 'TARGET_DFC_MI

13:36:44 | INFO     | Bloco 1 salvo


  ✅ Figura salva: b1_composicao_dataset.png


## Bloco 2 — Distribuição dos Targets

In [ ]:
print("\n" + "="*70)
print("  BLOCO 2 — Distribuição dos Targets (justifica log-transform)")
print("="*70)

fig, axes = plt.subplots(len(TARGETS), 3, figsize=(16, 4*len(TARGETS)))
if len(TARGETS) == 1:
    axes = axes.reshape(1, -1)

TARGET_LABELS = {
    'TARGET_DRE_3.01': 'Receita Líquida (R$ mil)',
    'TARGET_DRE_3.11': 'Lucro Líquido (R$ mil)',
    'TARGET_EBITDA':   'EBITDA (R$ mil)',
}

for i, t in enumerate(TARGETS):
    s = dataset[t].dropna()
    label = TARGET_LABELS.get(t, t)

    # Histograma original
    axes[i,0].hist(s, bins=40, color='#1f4e79', alpha=0.8, edgecolor='white', lw=0.5)
    axes[i,0].set_title(f'{label}\nOriginal (skew={s.skew():.2f})', fontsize=10)
    axes[i,0].set_xlabel('R$ mil')
    axes[i,0].set_ylabel('Frequência')

    # Histograma log1p (apenas positivos)
    s_pos = s[s > 0]
    s_log = np.log1p(s_pos)
    axes[i,1].hist(s_log, bins=40, color='#ed7d31', alpha=0.8, edgecolor='white', lw=0.5)
    axes[i,1].set_title(f'Log1p transform\nskew={s_log.skew():.2f} | '
                         f'{len(s_pos)/len(s):.0%} valores positivos', fontsize=10)
    axes[i,1].set_xlabel('log(1 + R$ mil)')

    # Boxplot por setor
    df_box = dataset[['SETOR', t]].dropna()
    setores_ord = df_box.groupby('SETOR')[t].median().sort_values(ascending=False).index
    data_box = [df_box[df_box['SETOR']==s][t].values for s in setores_ord]
    bp = axes[i,2].boxplot(data_box, labels=setores_ord, patch_artist=True,
                            medianprops={'color':'black','lw':2})
    for patch, setor in zip(bp['boxes'], setores_ord):
        patch.set_facecolor(CORES_SETOR.get(setor, '#999'))
    axes[i,2].set_title(f'Distribuição por Setor', fontsize=10)
    axes[i,2].set_ylabel('R$ mil')
    axes[i,2].tick_params(axis='x', rotation=30)
    axes[i,2].set_yscale('symlog')

    print(f"  {t}: n={len(s):,}  mean={s.mean():,.0f}  "
          f"std={s.std():,.0f}  skew={s.skew():.2f}  kurt={s.kurtosis():.2f}")

plt.suptitle('Distribuição dos Targets — Original vs Log-Transform vs Setor',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_EDA / 'b2_distribuicao_targets.png', dpi=150, bbox_inches='tight')
plt.close()
logger.info("Bloco 2 salvo")
print("  ✅ Figura salva: b2_distribuicao_targets.png")



  BLOCO 2 — Distribuição dos Targets (justifica log-transform)
  TARGET_DRE_3.01_ITR_T1: n=1,006  mean=22,280,343  std=46,424,960  skew=4.41  kurt=25.86
  TARGET_DRE_3.01_ITR_T2: n=958  mean=25,784,496  std=50,849,451  skew=3.89  kurt=19.54
  TARGET_DRE_3.01_ITR_T3: n=933  mean=29,336,772  std=58,037,661  skew=3.82  kurt=18.19
  TARGET_DRE_3.01_DFP: n=977  mean=52,415,617  std=95,576,583  skew=3.23  kurt=11.98
  TARGET_DRE_3.11_ITR_T1: n=1,006  mean=2,364,036  std=11,158,134  skew=5.74  kurt=54.84
  TARGET_DRE_3.11_ITR_T2: n=958  mean=2,812,932  std=12,364,097  skew=5.26  kurt=43.00
  TARGET_DRE_3.11_ITR_T3: n=933  mean=3,299,154  std=13,976,527  skew=5.30  kurt=40.42
  TARGET_DRE_3.11_DFP: n=977  mean=5,482,987  std=20,821,197  skew=5.64  kurt=37.15
  TARGET_EBITDA_ITR_T1: n=1,006  mean=9,802,353  std=13,444,955  skew=1.82  kurt=2.03
  TARGET_EBITDA_ITR_T2: n=958  mean=10,736,238  std=13,758,245  skew=1.67  kurt=1.51
  TARGET_EBITDA_ITR_T3: n=933  mean=11,563,689  std=14,078,103  ske

13:37:06 | INFO     | Bloco 2 salvo


  ✅ Figura salva: b2_distribuicao_targets.png


## Bloco 3 — Evolução Temporal dos KPIs por Setor

In [ ]:
print("\n" + "="*70)
print("  BLOCO 3 — Evolução Temporal dos KPIs por Setor")
print("="*70)

kpis_plot = ['margem_ebitda','margem_liquida','roe','liquidez_corrente',
             'endividamento','cobertura_juros']
kpis_plot = [k for k in kpis_plot if k in dataset.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

df_dfp = dataset[dataset['ORIGEM'] == 'DFP'].copy()
df_dfp['ANO'] = df_dfp['ANO'].astype(int)

for idx, kpi in enumerate(kpis_plot):
    ax = axes[idx]
    for setor, cor in CORES_SETOR.items():
        sub = df_dfp[df_dfp['SETOR'] == setor].groupby('ANO')[kpi].median()
        if not sub.empty:
            ax.plot(sub.index, sub.values, 'o-', color=cor,
                    label=setor, lw=2, ms=5, alpha=0.9)
    ax.set_title(kpi.replace('_',' ').title(), fontsize=11, fontweight='bold')
    ax.set_xlabel('Ano')
    ax.grid(alpha=0.3)
    ax.tick_params(axis='x', rotation=45)
    if idx == 0:
        ax.legend(fontsize=8, loc='upper left')

# Remover eixos extras
for j in range(len(kpis_plot), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Evolução Temporal de KPIs por Setor — Mediana (DFP Anual)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_EDA / 'b3_evolucao_kpis_setor.png', dpi=150, bbox_inches='tight')
plt.close()
logger.info("Bloco 3 salvo")
print("  ✅ Figura salva: b3_evolucao_kpis_setor.png")



  BLOCO 3 — Evolução Temporal dos KPIs por Setor


13:37:07 | INFO     | Bloco 3 salvo


  ✅ Figura salva: b3_evolucao_kpis_setor.png


## Bloco 4 — Heatmap de Correlação das Features

In [ ]:
print("\n" + "="*70)
print("  BLOCO 4 — Heatmap de Correlação Features × Targets")
print("="*70)

feat_num = [f for f in FEATURES if f in dataset.columns
            and not f.startswith('setor_') and f != 'flag_dfp']

fig, axes = plt.subplots(1, len(TARGETS), figsize=(6*len(TARGETS), max(8, len(feat_num)//3)))

for i, t in enumerate(TARGETS):
    df_c = dataset[feat_num + [t]].dropna()
    corr = df_c[feat_num].corrwith(df_c[t]).sort_values()
    colors = ['#c0392b' if v < 0 else '#1f4e79' for v in corr.values]
    axes[i].barh(range(len(corr)), corr.values, color=colors, alpha=0.85)
    axes[i].set_yticks(range(len(corr)))
    axes[i].set_yticklabels(corr.index, fontsize=9)
    axes[i].axvline(0, color='black', lw=0.8)
    axes[i].axvline(0.1,  color='gray', lw=0.8, ls='--', alpha=0.5)
    axes[i].axvline(-0.1, color='gray', lw=0.8, ls='--', alpha=0.5)
    axes[i].set_title(f'Correlação com\n{t.replace("TARGET_","")}',
                       fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Pearson r')
    axes[i].grid(axis='x', alpha=0.3)
    # Anotar os valores
    for j, v in enumerate(corr.values):
        axes[i].text(v + 0.005*np.sign(v) if v != 0 else 0.005,
                     j, f'{v:.2f}', va='center', fontsize=7.5)
    print(f"  {t}: top3={list(corr.abs().sort_values(ascending=False).head(3).index)}")

plt.suptitle('Correlação de Pearson — Features × Targets',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_EDA / 'b4_correlacao_features.png', dpi=150, bbox_inches='tight')
plt.close()
logger.info("Bloco 4 salvo")
print("  ✅ Figura salva: b4_correlacao_features.png")



  BLOCO 4 — Heatmap de Correlação Features × Targets
  TARGET_DRE_3.01_ITR_T1: top3=['TARGET_DRE_3.01_ITR_T2_lag1', 'TARGET_DRE_3.01_ITR_T1_lag4', 'TARGET_DRE_3.01_ITR_T3_lag2']
  TARGET_DRE_3.01_ITR_T2: top3=['TARGET_DRE_3.01_ITR_T3_lag1', 'TARGET_DRE_3.01_ITR_T2_lag4', 'TARGET_DRE_3.01_ITR_T3_roll2_mean']
  TARGET_DRE_3.01_ITR_T3: top3=['TARGET_DRE_3.01_ITR_T3_lag4', 'TARGET_DRE_3.01_ITR_T1_roll2_mean', 'TARGET_DRE_3.01_ITR_T1_lag2']
  TARGET_DRE_3.01_DFP: top3=['TARGET_DRE_3.01_ITR_T3_roll4_mean', 'TARGET_DRE_3.01_ITR_T2_roll4_mean', 'TARGET_DRE_3.01_DFP_lag1']
  TARGET_DRE_3.11_ITR_T1: top3=['TARGET_DRE_3.11_ITR_T2_lag1', 'TARGET_DRE_3.11_ITR_T3_lag2', 'TARGET_DRE_3.11_ITR_T2_roll2_mean']
  TARGET_DRE_3.11_ITR_T2: top3=['TARGET_DRE_3.11_ITR_T3_lag1', 'TARGET_DRE_3.11_ITR_T3_roll2_mean', 'TARGET_DRE_3.11_ITR_T3_roll4_mean']
  TARGET_DRE_3.11_ITR_T3: top3=['TARGET_DRE_3.11_ITR_T3_lag1', 'TARGET_DFC_MI_6.01_ITR_T3_lag4', 'TARGET_DFC_MI_6.01_ITR_T1_roll2_mean']
  TARGET_DRE_3.11_DFP: 

13:40:06 | INFO     | Bloco 4 salvo


  ✅ Figura salva: b4_correlacao_features.png


## Bloco 5 — Matriz de Missingness

In [ ]:
print("\n" + "="*70)
print("  BLOCO 5 — Missingness das Features por Empresa e Ano")
print("="*70)

# Missingness das features numéricas do modelo
df_miss = dataset[FEATURES + TARGETS].isnull().mean().sort_values(ascending=False)
df_miss = df_miss[df_miss > 0]

if not df_miss.empty:
    fig, ax = plt.subplots(figsize=(12, max(4, len(df_miss)*0.4)))
    colors = ['#c0392b' if v > 0.3 else '#f39c12' if v > 0.1 else '#27ae60'
              for v in df_miss.values]
    ax.barh(df_miss.index, df_miss.values * 100, color=colors, alpha=0.85)
    ax.set_xlabel('% de valores ausentes')
    ax.set_title('Missingness das Features e Targets no Dataset Preparado',
                 fontsize=12, fontweight='bold')
    ax.axvline(30, color='red', ls='--', lw=1, label='Limiar 30%')
    ax.axvline(10, color='orange', ls='--', lw=1, label='Limiar 10%')
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    for j, v in enumerate(df_miss.values):
        ax.text(v*100 + 0.3, j, f'{v:.1%}', va='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(PASTA_EDA / 'b5_missingness.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Colunas com NaN: {len(df_miss)}")
    print(f"  Maior missingness: {df_miss.index[0]} = {df_miss.iloc[0]:.1%}")
else:
    print("  ✅ Sem valores ausentes nas features e targets")
    # Criar figura simples
    fig, ax = plt.subplots(figsize=(8,3))
    ax.text(0.5, 0.5, '✅ Sem valores ausentes\nnas features e targets',
            ha='center', va='center', fontsize=14, transform=ax.transAxes)
    ax.axis('off')
    plt.savefig(PASTA_EDA / 'b5_missingness.png', dpi=150)
    plt.close()

logger.info("Bloco 5 salvo")
print("  ✅ Figura salva: b5_missingness.png")



  BLOCO 5 — Missingness das Features por Empresa e Ano


13:40:15 | INFO     | Bloco 5 salvo


  Colunas com NaN: 394
  Maior missingness: retorno_12m_roll4_mean = 30.1%
  ✅ Figura salva: b5_missingness.png


## Bloco 6 — Cobertura das Variáveis Macro

In [ ]:
print("\n" + "="*70)
print("  BLOCO 6 — Cobertura das Variáveis Macroeconômicas por Ano")
print("="*70)

if MACRO_COLS:
    df_macro_cob = (dataset.groupby('ANO')[MACRO_COLS]
                    .apply(lambda g: g.notna().mean())
                    .reset_index())
    df_macro_cob['ANO'] = df_macro_cob['ANO'].astype(int)

    fig, ax = plt.subplots(figsize=(14, 6))
    x     = np.arange(len(df_macro_cob))
    width = 0.8 / max(len(MACRO_COLS), 1)
    colors_m = plt.cm.tab10(np.linspace(0, 0.8, len(MACRO_COLS)))

    for j, col in enumerate(MACRO_COLS):
        vals = df_macro_cob[col].values * 100
        offset = (j - len(MACRO_COLS)/2 + 0.5) * width
        ax.bar(x + offset, vals, width*0.9, label=col.replace('macro_',''),
               color=colors_m[j], alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(df_macro_cob['ANO'].astype(str), rotation=45)
    ax.set_ylabel('Cobertura (%)')
    ax.set_ylim(0, 110)
    ax.set_title('Cobertura das Variáveis Macroeconômicas por Ano',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, bbox_to_anchor=(1.01, 1), loc='upper left')
    ax.axhline(100, color='green', ls='--', lw=1, alpha=0.5)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(PASTA_EDA / 'b6_cobertura_macro.png', dpi=150, bbox_inches='tight')
    plt.close()

    # Tabela de cobertura
    print("  Cobertura média por variável macro:")
    for col in MACRO_COLS:
        cob = dataset[col].notna().mean()
        print(f"    {col:<30} {cob:.0%}")
else:
    print("  Sem variáveis macro no dataset")

logger.info("Bloco 6 salvo")
print("  ✅ Figura salva: b6_cobertura_macro.png")



  BLOCO 6 — Cobertura das Variáveis Macroeconômicas por Ano


13:40:16 | INFO     | Bloco 6 salvo


  Cobertura média por variável macro:
    macro_selic                    100%
    macro_ipca                     100%
    macro_cambio                   100%
    macro_pib_tri                  100%
    macro_vol_brasil               100%
    retorno_12m                    72%
    volatilidade_60d               79%
    macro_selic_lag1               98%
    macro_selic_lag2               95%
    macro_selic_lag4               90%
    macro_selic_diff1              98%
    macro_selic_growth1            98%
    macro_selic_roll2_mean         95%
    macro_selic_roll4_mean         95%
    macro_ipca_lag1                98%
    macro_ipca_lag2                95%
    macro_ipca_lag4                90%
    macro_ipca_diff1               98%
    macro_ipca_growth1             98%
    macro_ipca_roll2_mean          95%
    macro_ipca_roll4_mean          95%
    macro_cambio_lag1              98%
    macro_cambio_lag2              95%
    macro_cambio_lag4              90%
    macro_cambio_diff

## Bloco 7 — Análise de Outliers por KPI e Setor

In [ ]:
print("\n" + "="*70)
print("  BLOCO 7 — Análise de Outliers por KPI e Setor (Winsorização 3×IQR)")
print("="*70)

kpis_outlier = ['margem_ebitda','roe','liquidez_corrente',
                'endividamento','cobertura_juros','giro_ativo']
kpis_outlier = [k for k in kpis_outlier if k in dataset.columns]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, kpi in enumerate(kpis_outlier):
    ax = axes[idx]
    data_setor = []
    labels_setor = []
    for setor in CORES_SETOR:
        sub = dataset[dataset['SETOR']==setor][kpi].dropna()
        if not sub.empty:
            data_setor.append(sub.values)
            labels_setor.append(setor)

    bp = ax.violinplot(data_setor, showmedians=True, showextrema=True)
    for j, (body, setor) in enumerate(zip(bp['bodies'], labels_setor)):
        body.set_facecolor(list(CORES_SETOR.values())[j % len(CORES_SETOR)])
        body.set_alpha(0.7)

    ax.set_xticks(range(1, len(labels_setor)+1))
    ax.set_xticklabels(labels_setor, rotation=30, fontsize=9)
    ax.set_title(kpi.replace('_',' ').title(), fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    # Contar outliers (fora de 3×IQR)
    total = dataset[kpi].dropna()
    q1, q3 = total.quantile(0.25), total.quantile(0.75)
    iqr = q3 - q1
    n_out = ((total < q1-3*iqr) | (total > q3+3*iqr)).sum()
    ax.set_xlabel(f'Outliers extremos (3×IQR): {n_out} ({n_out/len(total):.1%})',
                  fontsize=8)

for j in range(len(kpis_outlier), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Distribuição de KPIs por Setor — Violin Plot',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_EDA / 'b7_outliers_kpi_setor.png', dpi=150, bbox_inches='tight')
plt.close()
logger.info("Bloco 7 salvo")
print("  ✅ Figura salva: b7_outliers_kpi_setor.png")



  BLOCO 7 — Análise de Outliers por KPI e Setor (Winsorização 3×IQR)


13:40:18 | INFO     | Bloco 7 salvo


  ✅ Figura salva: b7_outliers_kpi_setor.png


## Bloco 8 — Cobertura Temporal por Empresa

In [ ]:
print("\n" + "="*70)
print("  BLOCO 8 — Cobertura Temporal por Empresa (DFP + ITR)")
print("="*70)

df_cob = (dataset.groupby(['NOME_CIA','SETOR'])
          .agg(
              Primeiro=('ANO','min'),
              Ultimo=('ANO','max'),
              N_DFP=('ORIGEM', lambda x: (x=='DFP').sum()),
              N_ITR=('ORIGEM', lambda x: (x=='ITR').sum()),
          ).reset_index()
          .sort_values(['SETOR','NOME_CIA']))
df_cob['Total'] = df_cob['N_DFP'] + df_cob['N_ITR']
df_cob['Anos'] = (df_cob['Ultimo'] - df_cob['Primeiro'] + 1).astype(int)

print(df_cob[['NOME_CIA','SETOR','Primeiro','Ultimo','N_DFP','N_ITR','Total']].to_string(index=False))

# Heatmap empresa × ano
pivot_cob = (dataset.groupby(['NOME_CIA','ANO']).size()
             .unstack(fill_value=0))
pivot_cob.columns = pivot_cob.columns.astype(int)

fig, ax = plt.subplots(figsize=(16, max(8, len(pivot_cob)*0.4)))
sns.heatmap(pivot_cob, ax=ax, cmap='YlOrRd', linewidths=0.3,
            linecolor='white', annot=True, fmt='d', annot_kws={'size':7},
            cbar_kws={'label':'Nº de observações'})
ax.set_title('Observações por Empresa e Ano (DFP + ITR)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Ano')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(PASTA_EDA / 'b8_cobertura_empresa_ano.png', dpi=150, bbox_inches='tight')
plt.close()
logger.info("Bloco 8 salvo")
print("  ✅ Figura salva: b8_cobertura_empresa_ano.png")



  BLOCO 8 — Cobertura Temporal por Empresa (DFP + ITR)
          NOME_CIA       SETOR  Primeiro  Ultimo  N_DFP  N_ITR  Total
     CSN Mineração Commodities      2015    2026     11     34     45
            Gerdau Commodities      2015    2026     11     34     45
            Klabin Commodities      2015    2026     11     34     45
            Suzano Commodities      2015    2026     11     34     45
              Vale Commodities      2015    2026     11     34     45
      CPFL Energia     Energia      2015    2026     11     34     45
      Engie Brasil     Energia      2015    2026     11     34     45
Equatorial Energia     Energia      2015    2026     11     34     45
         ISA CTEEP     Energia      2015    2026     11     34     45
             Taesa     Energia      2015    2026     11     34     45
         Petrobras    Petróleo      2015    2026     11     34     45
              Prio    Petróleo      2015    2026     11     34     45
            Raízen    Petróleo    

13:40:19 | INFO     | Bloco 8 salvo


  ✅ Figura salva: b8_cobertura_empresa_ano.png


## Bloco 9 — Estatísticas Descritivas Consolidadas

In [ ]:
print("\n" + "="*70)
print("  BLOCO 9 — Estatísticas Descritivas para o TCC")
print("="*70)

kpis_desc = [k for k in KPIS if k in dataset.columns]
desc = dataset[kpis_desc].describe().T
desc['skew'] = dataset[kpis_desc].skew()
desc['cv']   = dataset[kpis_desc].std() / dataset[kpis_desc].mean().abs()

print("\nKPIs — Estatísticas Descritivas:")
print(desc[['count','mean','std','min','50%','max','skew']].round(3).to_string())

# Figura: heatmap de estatísticas descritivas normalizadas
fig, axes = plt.subplots(1, 2, figsize=(18, max(6, len(kpis_desc)*0.4)))

# Médias normalizadas por setor (z-score)
medias = (dataset.groupby('SETOR')[kpis_desc].median()
          .apply(lambda col: (col - col.mean()) / (col.std() + 1e-9)))
sns.heatmap(medias.T, ax=axes[0], cmap='RdYlGn', center=0,
            linewidths=0.3, linecolor='white',
            annot=True, fmt='.1f', annot_kws={'size':7},
            cbar_kws={'label':'Z-score da mediana setorial'})
axes[0].set_title('Perfil Financeiro por Setor (Z-score)',
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('')

# Cobertura dos KPIs
cob_kpi = dataset[kpis_desc].notna().mean().sort_values()
colors_cob = ['#27ae60' if v >= 0.9 else '#f39c12' if v >= 0.7 else '#c0392b'
              for v in cob_kpi.values]
axes[1].barh(cob_kpi.index, cob_kpi.values * 100, color=colors_cob, alpha=0.85)
axes[1].set_xlabel('Cobertura (%)')
axes[1].set_title('Cobertura dos KPIs no Dataset', fontsize=11, fontweight='bold')
axes[1].axvline(90, color='green', ls='--', lw=1, label='90%')
axes[1].axvline(70, color='orange', ls='--', lw=1, label='70%')
axes[1].legend(fontsize=8)
axes[1].grid(axis='x', alpha=0.3)
for j, v in enumerate(cob_kpi.values):
    axes[1].text(v*100 + 0.3, j, f'{v:.0%}', va='center', fontsize=8)

plt.suptitle('Análise Descritiva do Dataset — Perfil Setorial e Cobertura',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_EDA / 'b9_descritivas_consolidadas.png', dpi=150, bbox_inches='tight')
plt.close()
logger.info("Bloco 9 salvo")
print("  ✅ Figura salva: b9_descritivas_consolidadas.png")



  BLOCO 9 — Estatísticas Descritivas para o TCC

KPIs — Estatísticas Descritivas:
                       count            mean             std             min            50%             max    skew
margem_bruta      1,031.0000          0.3700          0.2050         -0.5340         0.3350          0.9640  0.2480
margem_ebit       1,031.0000          0.2120          0.2270         -0.5270         0.1400          0.8660  1.2320
margem_liquida    1,031.0000          0.1180          0.1710         -0.4110         0.0840          0.5930  0.8080
margem_ebitda     1,031.0000          1.0380          0.9250          0.0680         0.7020          4.6500  1.4730
roe               1,031.0000          0.0900          0.1240         -0.3500         0.0760          0.5260  0.2570
roa               1,031.0000          0.0360          0.0450         -0.1490         0.0280          0.2200  0.4340
liquidez_corrente 1,031.0000          2.0450          0.9110          0.4120         1.8290          5.76

13:40:21 | INFO     | Bloco 9 salvo


  ✅ Figura salva: b9_descritivas_consolidadas.png


## Relatório Final

## Bloco 10 — Painel Longitudinal por Empresa (dados históricos)

Visão individual de cada empresa ao longo do tempo — sem predição.
Complementa os blocos 3 e 9 (nível setorial) com a granularidade necessária
para a banca avaliar a qualidade dos dados e a evolução real dos indicadores.

**Saídas geradas:**
- `b10_painel_kpis_empresa.csv`          — empresa × ano × 10 KPIs
- `b10_painel_targets_empresa.csv`       — empresa × ano × Receita / Lucro / EBITDA
- `b10_longitudinal_kpis_{setor}.png`    — grid empresa × KPI (série temporal)
- `b10_longitudinal_targets_{setor}.png` — Receita / Lucro / EBITDA por empresa (barras)

In [ ]:
print("\n" + "="*70)
print("  BLOCO 10 — Painel Longitudinal por Empresa (dados históricos)")
print("="*70)

# ── Configuração ──────────────────────────────────────────────────────────────
KPI_PAINEL = [k for k in [
    'margem_ebitda', 'margem_liquida', 'roe', 'roa',
    'liquidez_corrente', 'liquidez_imediata',
    'endividamento', 'cobertura_juros',
    'giro_ativo', 'fco_receita',
] if k in dataset.columns]

BASES_HIST = {b: l for b, l in {
    'DRE_3.01': 'Receita Líquida',
    'DRE_3.11': 'Lucro Líquido',
    'EBITDA'  : 'EBITDA',
}.items() if b in ds_raw.columns}

# Filtra apenas DFP para séries anuais limpas
df_dfp = dataset[dataset['ORIGEM'] == 'DFP'].copy()
df_dfp['ANO'] = df_dfp['ANO'].astype(int)

ds_dfp = ds_raw[ds_raw['ORIGEM'] == 'DFP'].copy() if 'ORIGEM' in ds_raw.columns else ds_raw.copy()
if 'ANO' not in ds_dfp.columns and 'DT_REFER' in ds_dfp.columns:
    ds_dfp['ANO'] = pd.to_datetime(ds_dfp['DT_REFER'], errors='coerce').dt.year
ds_dfp['ANO'] = ds_dfp['ANO'].astype('Int64')

# ── CSV 1: KPIs por empresa × ano ─────────────────────────────────────────────
rows_kpi = []
for (nome, ano), grp in df_dfp.groupby(['NOME_CIA', 'ANO']):
    row = {'empresa': nome, 'setor': grp['SETOR'].iloc[0], 'ano': int(ano)}
    for k in KPI_PAINEL:
        vals = grp[k].dropna()
        row[k] = float(vals.iloc[-1]) if not vals.empty else None
    rows_kpi.append(row)

df_pk = pd.DataFrame(rows_kpi).sort_values(['setor','empresa','ano'])
df_pk.to_csv(PASTA_EDA / 'b10_painel_kpis_empresa.csv', index=False)
print(f"  ✅ b10_painel_kpis_empresa.csv ({len(df_pk)} linhas, {df_pk['empresa'].nunique()} empresas)")

# ── CSV 2: Receita / Lucro / EBITDA por empresa × ano ────────────────────────
rows_tgt = []
for (cnpj, ano), grp in ds_dfp.groupby(['CNPJ_CIA', 'ANO']):
    nome  = grp['NOME_CIA'].iloc[0] if 'NOME_CIA' in grp.columns else str(cnpj)
    setor = grp['SETOR'].iloc[0]    if 'SETOR'   in grp.columns else ''
    row   = {'empresa': nome, 'setor': setor, 'ano': int(ano)}
    for b in BASES_HIST:
        if b in grp.columns:
            vals = grp[b].dropna()
            row[b] = float(vals.iloc[-1]) if not vals.empty else None
    rows_tgt.append(row)

df_pt = pd.DataFrame(rows_tgt).sort_values(['setor','empresa','ano'])
df_pt.to_csv(PASTA_EDA / 'b10_painel_targets_empresa.csv', index=False)
print(f"  ✅ b10_painel_targets_empresa.csv ({len(df_pt)} linhas)")

# ── Figuras por setor ─────────────────────────────────────────────────────────
setores = sorted(df_dfp['SETOR'].dropna().unique())

for setor in setores:
    empresas = sorted(df_dfp[df_dfp['SETOR'] == setor]['NOME_CIA'].unique())
    n_emp = len(empresas)
    n_kpi = len(KPI_PAINEL)
    if n_emp == 0 or n_kpi == 0:
        continue
    cor_s = CORES_SETOR.get(setor, '#3498db')

    # ── Fig A: KPIs (linhas de tempo por empresa × KPI) ──────────────────────
    fig, axes = plt.subplots(n_emp, n_kpi,
                              figsize=(3.2 * n_kpi, 3.0 * n_emp),
                              squeeze=False)
    for i_e, empresa in enumerate(empresas):
        df_e = df_dfp[df_dfp['NOME_CIA'] == empresa].sort_values('ANO')
        for i_k, kpi in enumerate(KPI_PAINEL):
            ax = axes[i_e][i_k]
            serie = df_e[['ANO', kpi]].dropna()
            if not serie.empty:
                ax.plot(serie['ANO'], serie[kpi],
                        'o-', color=cor_s, lw=1.8, ms=3.5)
                ax.axvspan(2020, 2021, alpha=0.10, color='gray')
                if kpi in ['margem_liquida','roe','roa','fco_receita']:
                    ax.axhline(0, color='#e74c3c', lw=0.8, ls='--', alpha=0.5)
                ax.tick_params(labelsize=5)
                ax.grid(alpha=0.2)
            else:
                ax.text(0.5, 0.5, 's/d', ha='center', va='center',
                        transform=ax.transAxes, fontsize=6, color='gray')
            if i_k == 0:
                ax.set_ylabel(empresa[:13] + ('…' if len(empresa)>13 else ''),
                              fontsize=6, fontweight='bold')
            if i_e == 0:
                ax.set_title(kpi.replace('_',' '), fontsize=7, fontweight='bold')
            if i_e < n_emp - 1:
                ax.set_xticklabels([])

    plt.suptitle(f'KPIs por Empresa — {setor} (DFP 2015–2025)',
                 fontsize=11, fontweight='bold', y=1.005)
    plt.tight_layout()
    fname = f'b10_longitudinal_kpis_{setor.replace(" ","_")}.png'
    plt.savefig(PASTA_EDA / fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ {fname}")

    # ── Fig B: Receita / Lucro / EBITDA (barras + linha) ─────────────────────
    n_b = len(BASES_HIST)
    if n_b == 0:
        continue
    fig2, axes2 = plt.subplots(n_emp, n_b,
                                figsize=(4.5 * n_b, 3.0 * n_emp),
                                squeeze=False)
    for i_e, empresa in enumerate(empresas):
        # Lookup CNPJ para buscar no ds_dfp (dados brutos, escala original)
        mask_n = ds_dfp['NOME_CIA'] == empresa if 'NOME_CIA' in ds_dfp.columns else pd.Series(False, index=ds_dfp.index)
        df_e_raw = ds_dfp[mask_n].sort_values('ANO') if mask_n.any() else pd.DataFrame()

        for i_b, (base, label_b) in enumerate(BASES_HIST.items()):
            ax = axes2[i_e][i_b]
            if not df_e_raw.empty and base in df_e_raw.columns:
                serie = df_e_raw[['ANO', base]].dropna().copy()
                serie['ANO'] = serie['ANO'].astype(int)
                if not serie.empty:
                    vals_bi = serie[base] / 1e6
                    cores_b = ['#e74c3c' if v < 0 else cor_s for v in vals_bi]
                    ax.bar(serie['ANO'], vals_bi, color=cores_b, alpha=0.75)
                    ax.plot(serie['ANO'], vals_bi, 'o-',
                            color='#2c3e50', lw=1, ms=3, alpha=0.6)
                    ax.axhline(0, color='#2c3e50', lw=0.6)
                    ax.axvspan(2020, 2021, alpha=0.09, color='gray')
                    ax.yaxis.set_major_formatter(
                        plt.FuncFormatter(lambda x, _: f'{x:.1f}'))
                    ax.tick_params(labelsize=5)
                    ax.grid(axis='y', alpha=0.2)
                else:
                    ax.text(0.5, 0.5, 's/d', ha='center', va='center',
                            transform=ax.transAxes, fontsize=6, color='gray')
            else:
                ax.text(0.5, 0.5, 's/d', ha='center', va='center',
                        transform=ax.transAxes, fontsize=6, color='gray')
            if i_b == 0:
                ax.set_ylabel(empresa[:13] + ('…' if len(empresa)>13 else ''),
                              fontsize=6, fontweight='bold')
            if i_e == 0:
                ax.set_title(f'{label_b}\n(R$ bilhões)', fontsize=7, fontweight='bold')
            if i_e < n_emp - 1:
                ax.set_xticklabels([])

    plt.suptitle(f'Receita / Lucro / EBITDA — {setor} (DFP 2015–2025)',
                 fontsize=11, fontweight='bold', y=1.005)
    plt.tight_layout()
    fname2 = f'b10_longitudinal_targets_{setor.replace(" ","_")}.png'
    plt.savefig(PASTA_EDA / fname2, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ {fname2}")

n_figs = len(list(PASTA_EDA.glob('b10_*.png')))
logger.info("Bloco 10 concluído: 2 CSVs + %d figuras", n_figs)
print(f"\n  ✅ Bloco 10 concluído — 2 CSVs + {n_figs} figuras em outputs/eda/")


  BLOCO 10 — Painel Longitudinal por Empresa (dados históricos)
  ✅ b10_painel_kpis_empresa.csv (254 linhas, 25 empresas)
  ✅ b10_painel_targets_empresa.csv (254 linhas)
  ✅ b10_longitudinal_kpis_Commodities.png
  ✅ b10_longitudinal_targets_Commodities.png
  ✅ b10_longitudinal_kpis_Energia.png
  ✅ b10_longitudinal_targets_Energia.png
  ✅ b10_longitudinal_kpis_Petróleo.png
  ✅ b10_longitudinal_targets_Petróleo.png
  ✅ b10_longitudinal_kpis_Tecnologia.png
  ✅ b10_longitudinal_targets_Tecnologia.png
  ✅ b10_longitudinal_kpis_Varejo.png


13:41:24 | INFO     | Bloco 10 concluído: 2 CSVs + 10 figuras


  ✅ b10_longitudinal_targets_Varejo.png

  ✅ Bloco 10 concluído — 2 CSVs + 10 figuras em outputs/eda/


In [ ]:
# Salvar relatório JSON com metadados da EDA
relatorio_eda = {
    'n_obs':        int(len(dataset)),
    'n_empresas':   int(dataset['NOME_CIA'].nunique()),
    'n_features':   int(len(FEATURES)),
    'n_targets':    int(len(TARGETS)),
    'anos':         sorted([int(a) for a in dataset['ANO'].dropna().unique()]),
    'origens':      {k: int(v) for k,v in dataset['ORIGEM'].value_counts().items()},
    'setores':      {k: int(v) for k,v in dataset['SETOR'].value_counts().items()},
    'macro_cobertura': {c: float(dataset[c].notna().mean()) for c in MACRO_COLS},
    'kpi_cobertura':   {k: float(dataset[k].notna().mean())
                        for k in KPIS if k in dataset.columns},
    'target_stats': {t: {
        'n':    int(dataset[t].notna().sum()),
        'mean': float(dataset[t].mean()),
        'std':  float(dataset[t].std()),
        'skew': float(dataset[t].skew()),
        'min':  float(dataset[t].min()),
        'max':  float(dataset[t].max()),
    } for t in TARGETS if t in dataset.columns},
    'figuras': [str(f.name) for f in sorted(PASTA_EDA.glob('*.png'))],
}

with open(PASTA_EDA / 'relatorio_eda.json', 'w', encoding='utf-8') as f:
    json.dump(relatorio_eda, f, indent=2, ensure_ascii=False, default=str)

print("\n" + "═"*70)
print("  RESUMO — Script 2.1 (EDA Pós-Processamento)")
print("═"*70)
print(f"  Dataset  : {len(dataset):,} obs × {dataset.shape[1]} colunas")
print(f"  Empresas : {dataset['NOME_CIA'].nunique()} / 25")
print(f"  Período  : {dataset['ANO'].min():.0f} – {dataset['ANO'].max():.0f}")
print(f"  Figuras  : {len(list(PASTA_EDA.glob('*.png')))} salvas em outputs/eda/")
print("═"*70)
print("  ✅ Pronto para o Script 3 (03_cvm_treino_V4.ipynb)")
print("═"*70)



══════════════════════════════════════════════════════════════════════
  RESUMO — Script 2.1 (EDA Pós-Processamento)
══════════════════════════════════════════════════════════════════════
  Dataset  : 1,031 obs × 992 colunas
  Empresas : 25 / 25
  Período  : 2015 – 2026
  Figuras  : 19 salvas em outputs/eda/
══════════════════════════════════════════════════════════════════════
  ✅ Pronto para o Script 3 (03_cvm_treino_V4.ipynb)
══════════════════════════════════════════════════════════════════════
